# EDA de agenda temática — NoticiasColombia (1980–2011)

Este notebook retoma el proyecto tras la observación del profesor: la técnica no debe
anteponerse a la pregunta. La pregunta de investigación (ver `CLAUDE.md`) es:

> **¿Qué tan estable es la agenda temática de la prensa colombiana entre 1980 y 2011 y
> cuándo se producen cambios significativos en dicha agenda?**

Este EDA corresponde a las **Etapas 1 y 2** del plan (auditoría del corpus + exploración
lingüística), y busca responder preguntas concretas que condicionan las decisiones de
las etapas siguientes (representación y modelado de tópicos):

1. ¿Hay desbalance de cobertura temporal entre fuentes? (sección 17 de `CLAUDE.md`)
2. ¿Qué resolución temporal (año / trimestre / mes) tiene suficiente volumen documental
   para modelar `P(z|t)` sin ruido excesivo? (sección 13)
3. ¿Hay anomalías de fecha que limiten esa resolución?
4. ¿Cómo se comporta el vocabulario (Zipf, curva de crecimiento) y qué implica para
   filtros de frecuencia en la etapa de representación? (sección 11.1)
5. ¿Cuánto contenido cuasi-duplicado (cables de agencia repetidos) queda tras la
   deduplicación exacta ya aplicada en `utils.load_corpus`?

**Política de salidas**: todas las figuras se muestran inline (`plt.show()`); solo se
persisten 2-3 tablas/resúmenes pequeños en `results/`, no una imagen por cada
combinación de fuente/década/bloque.

**Nota de ejecución**: pensado para correr en Google Colab (instala automáticamente
`spacy`, `nltk` y `pyarrow` vía `utils.setup()`). En máquina local basta con tener esas
dependencias instaladas.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from utils import load_corpus, results_dir, setup

setup()

df = load_corpus()
RESULTS = results_dir()

print(df.shape)
df.head(3)

## 1. Cobertura temporal por fuente

`CLAUDE.md` (sección 17) exige, antes de comparar fuentes, reportar (1) documentos por
medio, (2) cobertura temporal de cada medio y (3) evaluar el desbalance — precisamente
porque El Tiempo domina el corpus y cualquier patrón temporal podría ser un artefacto
de qué medio está presente en qué año, no un cambio real de agenda.

In [ ]:
anio = df["fecha"].dt.year

resumen_fuentes = pd.DataFrame({
    "documentos": df["fuente"].value_counts(),
    "pct": df["fuente"].value_counts(normalize=True).mul(100).round(1),
})
resumen_fuentes

In [ ]:
cobertura = pd.DataFrame({
    "documentos": df.groupby("fuente", observed=True).size(),
    "anio_min": df.groupby("fuente", observed=True)["fecha"].min().dt.year,
    "anio_max": df.groupby("fuente", observed=True)["fecha"].max().dt.year,
    "n_anios_activos": df.assign(anio=anio).groupby("fuente", observed=True)["anio"].nunique(),
})

anios_totales_periodo = int(anio.max() - anio.min() + 1)
cobertura["anios_totales_periodo"] = anios_totales_periodo
cobertura["pct_anios_cubiertos"] = (
    cobertura["n_anios_activos"] / anios_totales_periodo * 100
).round(1)

cobertura

La tabla anterior dice *si* cada fuente cubre el período; el heatmap siguiente dice
*cuánto* aporta cada una año a año. El panel derecho (normalizado por fila) es el que
importa para no confundir "cambio de agenda" con "cambio de qué medio está
disponible" ese año.

In [ ]:
tab = pd.crosstab(anio, df["fuente"])
tab_pct_fila = tab.div(tab.sum(axis=1), axis=0).mul(100)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

im0 = axes[0].imshow(tab.values, aspect="auto", cmap="viridis")
axes[0].set_yticks(range(len(tab.index)))
axes[0].set_yticklabels(tab.index)
axes[0].set_xticks(range(len(tab.columns)))
axes[0].set_xticklabels(tab.columns, rotation=45, ha="right")
axes[0].set_title("Documentos por año y fuente (conteo)")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(tab_pct_fila.values, aspect="auto", cmap="viridis")
axes[1].set_yticks(range(len(tab_pct_fila.index)))
axes[1].set_yticklabels(tab_pct_fila.index)
axes[1].set_xticks(range(len(tab_pct_fila.columns)))
axes[1].set_xticklabels(tab_pct_fila.columns, rotation=45, ha="right")
axes[1].set_title("% de cada año aportado por fuente")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

plt.tight_layout()
plt.show()

tab.to_csv(RESULTS / "cobertura_anio_fuente.csv")
print(f"guardado: {RESULTS / 'cobertura_anio_fuente.csv'}")

## 2. Resolución temporal: ¿año, trimestre o mes?

`CLAUDE.md` (sección 13) pide decidir esto empíricamente viendo volumen documental por
período, no por preferencia a priori. Construimos los tres calendarios completos
(incluyendo períodos con **cero** documentos, que `value_counts()` por sí solo no
muestra) para ver dónde empieza a aparecer ruido por escasez de datos.

In [ ]:
por_anio = anio.value_counts().sort_index()

rango_q = pd.period_range(
    df["fecha"].dt.to_period("Q").min(), df["fecha"].dt.to_period("Q").max(), freq="Q"
)
por_trimestre = (
    df["fecha"].dt.to_period("Q").value_counts().reindex(rango_q, fill_value=0).sort_index()
)

rango_m = pd.period_range(
    df["fecha"].dt.to_period("M").min(), df["fecha"].dt.to_period("M").max(), freq="M"
)
por_mes = (
    df["fecha"].dt.to_period("M").value_counts().reindex(rango_m, fill_value=0).sort_index()
)

print(f"Años: {len(por_anio)} períodos | media {por_anio.mean():.0f} docs | min {por_anio.min()} | max {por_anio.max()}")
print(
    f"Trimestres: {len(por_trimestre)} períodos | media {por_trimestre.mean():.1f} docs "
    f"| {(por_trimestre == 0).sum()} vacíos ({(por_trimestre == 0).mean():.1%})"
)
print(
    f"Meses: {len(por_mes)} períodos | media {por_mes.mean():.1f} docs "
    f"| {(por_mes == 0).sum()} vacíos ({(por_mes == 0).mean():.1%})"
)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9))

por_anio.plot.bar(ax=axes[0], color="#444", width=0.85)
axes[0].set_title("Documentos por año")
axes[0].set_ylabel("docs")

axes[1].plot(por_trimestre.index.to_timestamp(), por_trimestre.values, color="#444", lw=1)
axes[1].set_title("Documentos por trimestre (incluye períodos vacíos)")
axes[1].set_ylabel("docs")

axes[2].plot(por_mes.index.to_timestamp(), por_mes.values, color="#444", lw=0.7)
axes[2].set_title("Documentos por mes (incluye períodos vacíos)")
axes[2].set_ylabel("docs")

plt.tight_layout()
plt.show()

**Guía de lectura** (criterio general; la resolución definitiva de la Entrega 2 es anual y se justifica en `05-eda-avanzado-dataset-medido.ipynb`, sección 0):

- Si el % de meses vacíos es alto, mes es una resolución demasiado fina para todo el
  período — puede seguir siendo válida solo para sub-períodos densos.
- Si trimestre tiene pocos o ningún período vacío pero mes sí, trimestre es candidato
  razonable para todo el período 1980–2011.
- Conviene cruzar esto con la sección 1: un período puede tener "suficientes"
  documentos en total pero estar dominado por una sola fuente, lo que limitaría un
  análisis por medio en ese período aunque no limite el análisis agregado.

## 3. Anomalías de fecha

`01-corpus.ipynb` ya notó que `fecha.dt.time` no es uniformemente cero, pero no llegó a
una conclusión. Si una fracción grande de documentos tiene hora `00:00:00`, es señal de
que esa fecha se imputó solo con año/mes (o año/mes/día sin hora real), lo cual no
afecta la resolución mensual/trimestral pero sí advierte contra usar granularidad
diaria.

In [ ]:
medianoche = df["fecha"].dt.time == pd.Timestamp("00:00:00").time()
pct_medianoche = medianoche.mean()

print(f"% de documentos con hora = 00:00:00: {pct_medianoche:.1%}")
print()
print("Top horas más frecuentes:")
print(df["fecha"].dt.time.value_counts().head(10))

In [ ]:
# ¿Está concentrado en alguna fuente, o es transversal?
pct_medianoche_por_fuente = (
    df.assign(medianoche=medianoche)
    .groupby("fuente", observed=True)["medianoche"]
    .mean()
    .mul(100)
    .round(1)
)
pct_medianoche_por_fuente

## 4. Longitud de documentos: por fuente y en el tiempo

Longitud global ya se calculó en `01-corpus.ipynb`; aquí la cruzamos con año y fuente,
porque un cambio estructural en la longitud de las noticias (p. ej. un medio que
empieza a publicar notas más cortas) puede confundirse con un cambio de agenda si no
se controla. Usamos el mismo conteo por regex de `01-corpus.ipynb` (rápido); el conteo
fino por tokenización de spaCy se hace en la sección 5.

In [ ]:
df["n_palabras"] = df["texto"].str.count(r"\b\w+\b")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

df.boxplot(column="n_palabras", by="fuente", ax=axes[0])
axes[0].set_yscale("log")
axes[0].set_title("Longitud por fuente (escala log)")
axes[0].set_ylabel("palabras")
fig.suptitle("")

long_por_anio = df.groupby(anio)["n_palabras"].median()
axes[1].plot(long_por_anio.index, long_por_anio.values, color="#444")
axes[1].set_title("Mediana de longitud por año")
axes[1].set_xlabel("año")
axes[1].set_ylabel("palabras (mediana)")

plt.tight_layout()
plt.show()

## 5. Vocabulario: Ley de Zipf y curva de crecimiento

Tokenización idéntica a `02-textometria.ipynb` (spaCy en blanco, sin componentes,
excluyendo puntuación/espacios) para mantener consistencia metodológica entre
notebooks. Esto tokeniza el corpus completo dos veces (frecuencias globales y curva de
crecimiento en orden aleatorio); puede tardar un par de minutos en Colab.

In [ ]:
from collections import Counter

import spacy

from utils import normalizar_texto

df["texto_rep"] = normalizar_texto(df["texto"])
nlp = spacy.blank("es")

freqs = Counter()
longitudes = []
for doc in nlp.pipe(df["texto_rep"], batch_size=200):
    toks = [t.lower_ for t in doc if not t.is_punct and not t.is_space]
    freqs.update(toks)
    longitudes.append(len(toks))

print(f"tokens: {sum(longitudes):,} | tipos: {len(freqs):,}")

In [ ]:
frecs_ordenadas = np.array(sorted(freqs.values(), reverse=True))
rangos = np.arange(1, len(frecs_ordenadas) + 1)
zipf_ideal = frecs_ordenadas[0] / rangos

fig, ax = plt.subplots(figsize=(7, 5))
ax.loglog(rangos, frecs_ordenadas, marker=".", linestyle="none", ms=2, color="#444", label="corpus")
ax.loglog(rangos, zipf_ideal, color="crimson", lw=1, label="Zipf ideal (f₁ / rango)")
ax.set_xlabel("rango (log)")
ax.set_ylabel("frecuencia (log)")
ax.set_title("Ley de Zipf — corpus completo")
ax.legend()
plt.tight_layout()
plt.show()

La curva de crecimiento se calcula en un **orden aleatorio** de documentos (no el orden
temporal del corpus) para que la curva refleje crecimiento de vocabulario por volumen
de texto, y no un artefacto del orden en que están guardados los documentos.

In [ ]:
rng = np.random.default_rng(42)
orden = rng.permutation(len(df))
textos_permutados = df["texto_rep"].to_numpy()[orden]

vistos = set()
tipos_acum = []
tokens_acum = []
total_tok = 0

for doc in nlp.pipe(textos_permutados, batch_size=200):
    toks = [t.lower_ for t in doc if not t.is_punct and not t.is_space]
    total_tok += len(toks)
    vistos.update(toks)
    tokens_acum.append(total_tok)
    tipos_acum.append(len(vistos))

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(tokens_acum, tipos_acum, color="#444")
ax.set_xlabel("tokens acumulados")
ax.set_ylabel("tipos únicos acumulados")
ax.set_title("Curva de crecimiento del vocabulario (orden aleatorio de documentos)")
plt.tight_layout()
plt.show()

**Lectura**: si la curva de crecimiento sigue subiendo casi linealmente al final (no se
aplana), el corpus todavía está "descubriendo" vocabulario nuevo al ritmo actual de
muestreo — relevante para decidir si un vocabulario mínimo de frecuencia (p. ej.
excluir hapax legomena) es razonable antes de vectorizar en la Etapa 3, sin necesidad
de justificarlo solo por "reducir dimensionalidad".

## 6. Chequeo rápido de cuasi-duplicados

`utils.load_corpus` ya elimina duplicados **exactos** (tras normalizar espacios y
mayúsculas). En prensa de esta época es común que el mismo cable de agencia se publique
con variaciones menores (encabezado, fecha de edición). Esto es solo un **chequeo
aproximado** con un único min-hash sobre shingles de 12 palabras — no una deduplicación
definitiva. Sirve para decidir si vale la pena invertir en un método más riguroso más
adelante, no para descartar documentos ahora. Puede tardar uno o dos minutos.

In [ ]:
import zlib


def min_shingle_hash(texto: str, k: int = 12) -> int:
    palabras = texto.split()
    n = len(palabras)
    if n < k:
        return zlib.crc32(texto.encode("utf-8"))
    mejor = None
    for i in range(n - k + 1):
        h = zlib.crc32(" ".join(palabras[i : i + k]).encode("utf-8"))
        if mejor is None or h < mejor:
            mejor = h
    return mejor


df["_minhash"] = df["texto_rep"].map(min_shingle_hash)

grupos = df.groupby("_minhash").size()
candidatos = grupos[grupos > 1]
n_involucrados = int(candidatos.sum())

print(f"Grupos con el mismo min-hash (posibles cuasi-duplicados): {len(candidatos)}")
print(f"Documentos involucrados: {n_involucrados} de {len(df)} ({n_involucrados / len(df):.2%})")

In [ ]:
# Inspección manual de un grupo candidato, para juzgar si el método está funcionando
# (falsos positivos posibles: un único hash es una señal ruidosa, no una prueba)
if len(candidatos) > 0:
    ejemplo_hash = candidatos.index[0]
    ejemplo = df.loc[df["_minhash"] == ejemplo_hash, ["id", "fuente", "fecha", "texto"]]
    for _, fila in ejemplo.iterrows():
        print(fila["id"], fila["fuente"], fila["fecha"])
        print(fila["texto"][:300])
        print("---")

df = df.drop(columns="_minhash")

## 7. Resumen ejecutable

Se genera dinámicamente a partir de lo calculado arriba (no son valores fijados de
antemano) y se guarda como referencia rápida para el informe.

In [ ]:
resumen = f"""
RESUMEN EDA — corpus NoticiasColombia ({len(df):,} documentos únicos, {df['fecha'].dt.year.min()}-{df['fecha'].dt.year.max()})

1) Cobertura por fuente
{resumen_fuentes.to_string()}

Cobertura temporal por fuente:
{cobertura.to_string()}

2) Resolución temporal
- Años: {len(por_anio)} períodos | min {por_anio.min()} docs | max {por_anio.max()} docs
- Trimestres vacíos: {(por_trimestre == 0).sum()} de {len(por_trimestre)} ({(por_trimestre == 0).mean():.1%})
- Meses vacíos: {(por_mes == 0).sum()} de {len(por_mes)} ({(por_mes == 0).mean():.1%})

3) Anomalías de fecha
- Documentos con hora 00:00:00: {pct_medianoche:.1%}

4) Vocabulario
- Tokens: {sum(longitudes):,} | Tipos: {len(freqs):,}

5) Cuasi-duplicados (chequeo rápido, min-hash de shingles de 12 palabras)
- Grupos candidatos: {len(candidatos)}
- Documentos involucrados: {n_involucrados:,} ({n_involucrados / len(df):.2%})
"""

print(resumen)
(RESULTS / "resumen_eda.txt").write_text(resumen)
print(f"guardado: {RESULTS / 'resumen_eda.txt'}")

## Próximos pasos (a decidir con base en los resultados de arriba, no antes)

- **Resolución temporal**: fijar año/trimestre/mes según la sección 2, documentando la
  justificación (sección 13 de `CLAUDE.md` exige esto explícitamente).
- **Fuente**: si el desbalance de la sección 1 es fuerte en ciertos años, decidir si el
  análisis por medio (sección 18) es viable solo para un sub-período.
- **Preprocesamiento para representación** (Etapa 3): definir un umbral de frecuencia
  mínima de vocabulario a partir de la curva de crecimiento (sección 5), no de forma
  arbitraria.
- **Cuasi-duplicados**: si el % de la sección 6 es material, evaluar un método más
  riguroso (p. ej. MinHash con varias funciones + LSH por bandas) antes de vectorizar,
  para no inflar artificialmente la prevalencia de ciertos tópicos en la Etapa 4.

Este notebook deliberadamente **no** entra en TF-IDF, lematización ni modelado de
tópicos — eso corresponde a las Etapas 3 y 4, una vez estas decisiones queden fijadas.

**Estado (Entrega 2)**: estos puntos se resolvieron así: resolución anual homogénea; análisis por fuente y con pruebas estratificadas; el chequeo de cuasi-duplicados queda como limitación (~1 %); la técnica pasa de señales léxicas (`04-divergencia-lexica.ipynb`) a tópicos NMF con modelo nulo (`06-agenda-topicos-cambio.ipynb`).
